# King County House Prices: the Talk

The essential steps and figures of `02_king_county_project_dominic.ipynb`, in the same order. Each station has a heading (what we do), the figure, and one **What we see** line to read out. All numbers and figures come from notebook 02; this notebook contains no code.

## 1 · Goal and metric: predict the sale price, judged by RMSE in dollars
*(notebook 02, step 1)*

**Key point:** We assume a bank lends money based on the estimated value. One large miss on a villa costs more than many small ones, and RMSE squares the errors, so large ones weigh more. MAE and R² are reported alongside.

## 2 · Split the data once, by house, not by sale
*(notebook 02, step 3)*

| | houses | sales |
|:--|--:|--:|
| training set | 17,136 | 17,270 |
| test set | 4,284 | 4,327 |

**Key point:** 176 houses were sold more than once. A row-wise split would put 63 of them into both sets, so the model would be tested on houses it had already seen with a price. Cross-validation uses `GroupKFold` for the same reason.

## 3 · Look at the target: how is the price distributed?
*(notebook 02, step 4)*

![Histogram of the sale price, strongly right-skewed with a long tail of expensive houses, next to the histogram of the log price, which is almost symmetric](assets/talk/talk_03_price_distribution.png)

**What we see:** The price is strongly right-skewed: median \$450,000, mean \$542,521, up to \$7.7 million (skew 4.20). After a log it is almost symmetric (0.44). The few expensive houses will drive the RMSE.

## 4 · Which features are related to the price? Correlation against mutual information
*(notebook 02, step 4)*

![Two bar charts per feature: Pearson and Spearman correlation with the price on the left, mutual information with the price on the right. Zipcode has a correlation near 0 but the longest mutual information bar](assets/talk/talk_04_correlation_and_mi.png)

**What we see:** `sqft_living` (Pearson 0.70) and `grade` (0.67) are the strongest. `zipcode` has a correlation of almost 0, but the **highest mutual information (0.41)**: zip codes are labels without an order, yet the location decides the price. So the zip code goes into the model one-hot encoded.

## 5 · Which features carry the same information? Pearson and Spearman heatmaps
*(notebook 02, step 4)*

![Heatmap of the Pearson correlations between all features; the strongest values form a cluster around sqft_living](assets/talk/talk_05_heatmap_pearson.png)

![Heatmap of the Spearman correlations between all features; sqft_lot and sqft_lot15 reach 0.92](assets/talk/talk_05_heatmap_spearman.png)

**What we see:** A **size cluster** around `sqft_living`: `sqft_above` 0.88, `grade`, `sqft_living15` and `bathrooms` each 0.76. Pearson measures straight lines, Spearman any rising order (ranks): for `sqft_lot` and `sqft_lot15` Spearman is much higher (0.92 against 0.78), because a few huge lots disturb the line but not the order.

## 6 · Which column can be calculated from the others? Variance inflation factor (VIF)
*(notebook 02, step 4)*

VIF = 1 / (1 − R²) when a feature is predicted from all the others:

| Columns kept | `sqft_living` | `sqft_above` | `sqft_basement` | highest VIF |
|:--|--:|--:|--:|--:|
| all three area columns | ∞ | ∞ | ∞ | **∞** |
| without `sqft_basement` | 8.7 | 6.9 | | 8.7 |
| without `sqft_above` | 6.1 | | (kept) | **6.1** |

**What we see:** `sqft_living = sqft_above + sqft_basement` exactly, so each of the three can be calculated from the other two (infinite VIF). Dropping `sqft_above` loses nothing and leaves the lowest VIF. The same holds for `yr_built = sale_year − age_at_sale` (VIF about 1.3 million), so it is dropped, too.

## 7 · Hand changes: fixed rules in `prepare()`, applied to training and test set alike
*(notebook 02, step 4)*

| # | Change | Why |
|:--|:--|:--|
| 1 | fill `sqft_basement` with `sqft_living − sqft_above` | exact, nothing to learn |
| 2 | `sale_year`, `sale_month` from `date` | the date as text is unusable |
| 3 | `age_at_sale`, at least 0 | 11 houses were sold before they were finished |
| 4 | `renovated` only if renovated **before** the sale | 5 renovations after the sale would be leakage |
| 5 | `has_basement` | 59% of the houses have none |
| 6 | drop ids, `date`, `sqft_above`, `yr_built`, `yr_renovated` | no information or redundant (station 6) |

**Key point:** Everything that learns from the data (imputing, `log1p` for the lot sizes, scaling, one-hot encoding) sits in the pipeline instead, so it is fitted on the training part only.

## 8 · Two baselines every model has to beat
*(notebook 02, step 5)*

| | **Mean baseline** | **Rule baseline** |
|:--|:--|:--|
| What it says | "Every house costs the average price." | "Every house costs \$245.10 per square foot of living area." |
| Prediction | \$542,521 for every house | \$245.10 × `sqft_living` |
| Learned on the training set | one number: the mean price | one number: `median(price / sqft_living)` |
| Question it answers | Does a model learn anything at all? | Is a model better than a rule of thumb? |
| In the figure | grey dashed line | red line through the origin |
| Train / test RMSE | \$372,183 / \$347,480 | \$268,971 / \$250,933 |

![Scatter plot of price against living area for a sample of sales, with the rule baseline as a red line through the origin and the mean baseline as a grey horizontal line](assets/talk/talk_08_baselines.png)

**What we see:** The RMSE of the mean baseline is the standard deviation of the price, so its R² is 0. The rule already explains almost half of the variation (R² 0.478). The spread around the red line grows with the size of the house, and small houses lie mostly above it: the line starts at \$0, although the lot and the location have a value of their own.

## 9 · Pipeline and a first Ridge: does the zip code add information?
*(notebook 02, step 6)*

![Horizontal bars of the cross-validated RMSE for the mean baseline, Ridge without zip code and Ridge with zip code, with a red dashed line for the rule baseline](assets/talk/talk_09_zipcode_cv.png)

**What we see:** Ridge with 89 features: \$205,443 without the zip code, **\$164,853 with it**. The zip code lowers the error by about \$40,600, nearly four times the fold standard deviation, so it adds information on top of size and quality.

## 10 · Tuning M2: Ridge on polynomial features, random search then grid search
*(notebook 02, step 7)*

![Random search for M2: cross-validated RMSE against alpha on a log scale, one line for degree 2 and one for degree 3; degree 2 is lowest at small alpha and rises steeply above alpha 100](assets/talk/talk_10_m2_tuning.png)

**What we see:** Degree 2 (blue) lowers the error to **\$133,103** (M1, plain Ridge: \$164,831); degree 3 (orange) stays above \$145,000 for every `alpha`, because with 1,539 features it fits noise. The narrow grid search around `alpha` 7.5 then gives an almost flat curve, so by the **one-standard-error rule** we take the strongest penalty within \$2,702 of the best: `alpha` 7.46.

## 11 · Choosing the best model by cross-validation, before the test
*(notebook 02, step 7)*

![Horizontal bars of the cross-validated RMSE with error bars for the mean baseline, Ridge, M1, M2 and M3, with the rule baseline as a red dashed line](assets/talk/talk_11_model_comparison.png)

**What we see:** M3 (M2 on `log(price)`) has the lowest mean, \$127,038, but its folds spread almost three times as much (sd \$16,933 against \$6,153). M2 lies within one standard error of M3 (\$16,933 / √5 ≈ \$7,573), so we choose the simpler and more stable **M2**.

## 12 · One test, all models
*(notebook 02, step 8)*

| Model | train RMSE | CV RMSE | test RMSE | test R² |
|:--|--:|--:|--:|--:|
| mean baseline | \$372,183 | \$371,855 | \$347,480 | −0.001 |
| rule baseline | \$268,971 | | \$250,933 | 0.478 |
| M1: Ridge, tuned | \$163,637 | \$164,831 | \$144,988 | 0.826 |
| **M2: polynomial Ridge (best)** | \$124,152 | \$133,103 | **\$119,652** | **0.881** |
| M3: M2 on `log(price)` | \$115,266 | \$127,038 | \$113,621 | 0.893 |

**What we see:** M2 **halves the error of the rule** (\$119,652 against \$250,933). Every model is better on the test set, even the mean baseline: the test set is simply easier, with fewer villas. R² for M2 is 0.889 on training and 0.881 on test, so it does not overfit. M3 is a bit better on the test, too, but the choice was made before, and choosing again would misuse the test set.

## 13 · Predicted against true price, for every model
*(notebook 02, step 8)*

![Grid of scatter plots of predicted against true price, training set in blue and test set in orange, for the two baselines, Ridge, M1, M2 and M3, each with a black diagonal for a perfect prediction](assets/talk/talk_13_predicted_vs_true.png)

**What we see:** The regression counterpart of a confusion matrix: on the diagonal the prediction is perfect. From the baselines to M2 and M3 the cloud closes in on the diagonal. On the right, the dots fall below it: expensive houses are predicted too low. Training and test look alike.

## 14 · Where is the best model wrong? Residuals of M2 on the test set
*(notebook 02, step 8)*

![Left: residuals of M2 against the predicted price, a funnel that widens above one million dollars. Right: mean residual and RMSE per price quarter, the most expensive quarter with by far the largest RMSE](assets/talk/talk_14_residuals.png)

**What we see:** A **funnel**: the errors grow with the price. The cheapest quarter is overestimated by \$20,093 on average, the most expensive underestimated by \$41,952: the model pulls towards the average. Relative to the price, it is 10-15% off everywhere. The 23 waterfront houses have an RMSE of \$447,214, and the largest 5% of the errors make up 61% of the squared error.

## 15 · Summary: what each step changed
*(notebook 02, summary)*

![Waterfall chart of the cross-validated RMSE from the mean baseline of 372 thousand dollars down to M2 at 133 thousand, with the steps rule, features, zip code, tuned alpha and polynomial features](assets/talk/talk_15_waterfall.png)

**What we see:** The rule removes the largest share (−\$103k), then the features (−\$63k), the zip code (−\$41k) and the polynomial features (−\$32k). **Tuning `alpha` brought nothing** (±\$0k): the gains came from the features, not from the penalty.

## 16 · Summary: where on the map the best model is wrong
*(notebook 02, summary)*

![Two maps of King County: left every test sale coloured by its residual, right one circle per zip code coloured by the mean residual and sized by the RMSE; the largest errors lie around Lake Washington](assets/talk/talk_16_error_map.png)

**What we see:** The large errors lie in north-west Seattle and around Lake Washington: Bellevue (98004), Madison Park (98112) and Medina (98039). The cheaper and more uniform south is predicted well: the model struggles where the prices are highest and most varied.

## 17 · Conclusion

- **Best model: M2**, Ridge on degree-2 polynomial features plus one-hot zip codes: **test RMSE \$119,652, R² 0.881**, half the error of the rule baseline, no overfitting.
- **What made the difference:** the location (zip code) and the polynomial features, not tuning `alpha`.
- **Limits:** luxury and waterfront houses are underestimated, because they are rare and the data lack what makes them special (plot, interior); the test set is easier than the training set; the log model M3 may be slightly better and is worth a closer look.